In [2]:
!nvidia-smi

Wed Jun 10 06:42:47 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.159.04             Driver Version: 580.159.04     CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  NVIDIA H100 80GB HBM3          On  |   00000000:DB:00.0 Off |                    0 |
| N/A   26C    P0             79W /  700W |       0MiB /  81559MiB |      0%      Default |
|                                         |                        |             Disabled |
+-----------------------------------------+-----

In [3]:
!pip install -q "ultralytics==8.4.62"

In [4]:
!pip install -q gdown

In [6]:
!gdown "14Z2n-BfgaKreJuSwR_Hdtm_ItsKXoVrN" -O /workspace/v4_real_final_20260610_yolo_export.tar.gz

Downloading...
From (original): https://drive.google.com/uc?id=14Z2n-BfgaKreJuSwR_Hdtm_ItsKXoVrN
From (redirected): https://drive.google.com/uc?id=14Z2n-BfgaKreJuSwR_Hdtm_ItsKXoVrN&confirm=t&uuid=cb6959b1-b504-4d87-a0fb-1250044c3f1d
To: /workspace/v4_real_final_20260610_yolo_export.tar.gz
100%|██████████████████████████████████████| 8.16G/8.16G [00:56<00:00, 145MB/s]


In [7]:
!ls -lh /workspace/v4_real_final_20260610_yolo_export.tar.gz

-rw-rw-rw- 1 root root 7.7G Jun 10 06:48 /workspace/v4_real_final_20260610_yolo_export.tar.gz


In [8]:
from pathlib import Path

for p in Path("/workspace").iterdir():
    size_gb = p.stat().st_size / 1024**3 if p.is_file() else 0
    print(p.name, f"{size_gb:.2f} GB" if p.is_file() else "DIR")

v4_real_final_20260610_yolo_export.tar.gz 7.65 GB
.cache DIR
.ipynb_checkpoints DIR
Untitled.ipynb 0.00 GB


In [9]:
from pathlib import Path
import tarfile

PROJECT = Path("/workspace/uavape_v4_model_benchmark")
DATA_ROOT = PROJECT / "datasets"
DATA_DIR = DATA_ROOT / "v4_real_final_20260610"
ARCHIVE = Path("/workspace/v4_real_final_20260610_yolo_export.tar.gz")

DATA_ROOT.mkdir(parents=True, exist_ok=True)

if DATA_DIR.exists():
    print("Dataset already exists:", DATA_DIR)
else:
    print("Extracting archive...")
    with tarfile.open(ARCHIVE, "r:gz") as tar:
        tar.extractall(DATA_ROOT)
    print("Done")

print("DATA_DIR exists:", DATA_DIR.exists())

Extracting archive...
Done
DATA_DIR exists: True


In [10]:
removed = 0
for p in DATA_DIR.rglob("._*"):
    p.unlink()
    removed += 1

print("Removed sidecar files:", removed)

Removed sidecar files: 2418


In [11]:
import yaml

yaml_path = DATA_DIR / "dataset.yaml"
data_yaml = yaml.safe_load(yaml_path.read_text())
data_yaml["path"] = str(DATA_DIR)

with open(yaml_path, "w") as f:
    yaml.safe_dump(data_yaml, f, sort_keys=False)

print(yaml_path.read_text())

path: /workspace/uavape_v4_model_benchmark/datasets/v4_real_final_20260610
train: images/train
val: images/val
test: images/test
nc: 2
names:
  0: vape
  1: lighter



In [12]:
from pathlib import Path
from collections import Counter

def count_split(split):
    img_dir = DATA_DIR / "images" / split
    lab_dir = DATA_DIR / "labels" / split

    images = sorted([
        p for p in img_dir.iterdir()
        if p.is_file()
        and p.suffix.lower() in {".jpg", ".jpeg", ".png"}
        and not p.name.startswith("._")
    ])

    labels = sorted([
        p for p in lab_dir.glob("*.txt")
        if p.is_file()
        and not p.name.startswith("._")
    ])

    class_counts = Counter()
    boxes = 0

    for label in labels:
        for line in label.read_text(encoding="utf-8").splitlines():
            if not line.strip():
                continue
            cls = int(line.split()[0])
            class_counts[cls] += 1
            boxes += 1

    missing_labels = [
        img.name for img in images
        if not (lab_dir / f"{img.stem}.txt").exists()
    ]

    missing_images = [
        lab.name for lab in labels
        if not (img_dir / f"{lab.stem}.jpg").exists()
    ]

    return {
        "split": split,
        "images": len(images),
        "labels": len(labels),
        "boxes": boxes,
        "vape_boxes": class_counts[0],
        "lighter_boxes": class_counts[1],
        "missing_labels": len(missing_labels),
        "missing_images": len(missing_images),
    }

for split in ["train", "val", "test"]:
    print(count_split(split))

{'split': 'train', 'images': 837, 'labels': 837, 'boxes': 1317, 'vape_boxes': 870, 'lighter_boxes': 447, 'missing_labels': 0, 'missing_images': 0}
{'split': 'val', 'images': 183, 'labels': 183, 'boxes': 303, 'vape_boxes': 203, 'lighter_boxes': 100, 'missing_labels': 0, 'missing_images': 0}
{'split': 'test', 'images': 184, 'labels': 184, 'boxes': 420, 'vape_boxes': 255, 'lighter_boxes': 165, 'missing_labels': 0, 'missing_images': 0}


In [13]:
!rm /workspace/v4_real_final_20260610_yolo_export.tar.gz

In [14]:
from ultralytics import YOLO
from pathlib import Path

PROJECT = Path("/workspace/uavape_v4_model_benchmark")
RUN = {"model": "yolo26s.pt", "imgsz": 640}

model = YOLO(RUN["model"])

results = model.train(
    data=str(yaml_path),
    project=str(PROJECT / "runs"),
    name=f'v4_{Path(RUN["model"]).stem}_img{RUN["imgsz"]}',
    imgsz=RUN["imgsz"],
    epochs=100,
    batch=32,
    device=0,
    workers=8,
    seed=42,
    deterministic=True,
    cos_lr=True,
    patience=100,
    exist_ok=True,
    plots=True,
)

WARNING ⚠️ user config directory '/root/.config/Ultralytics' is not writable, using '/tmp/Ultralytics'. Set YOLO_CONFIG_DIR to override.
Creating new Ultralytics Settings v0.0.6 file ✅ 
View Ultralytics Settings with 'yolo settings' or at '/tmp/Ultralytics/settings.json'
Update Settings with 'yolo settings key=value', i.e. 'yolo settings runs_dir=path/to/dir'. For help see https://docs.ultralytics.com/quickstart/#ultralytics-settings.
New https://pypi.org/project/ultralytics/8.4.63 available 😃 Update with 'pip install -U ultralytics'
Ultralytics 8.4.62 🚀 Python-3.12.3 torch-2.8.0+cu128 CUDA:0 (NVIDIA H100 80GB HBM3, 81079MiB)
engine/trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=32, bgr=0.0, box=7.5, cache=False, cfg=None, classes=None, close_mosaic=10, cls=0.5, cls_pw=0.0, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=True, cutmix=0.0, data=/workspace/uavape_v4_model_benchmark/datasets/v4_real_final_20260610/da

In [17]:
import pandas as pd
from pathlib import Path

run_dir = Path("/workspace/uavape_v4_model_benchmark/runs/v4_yolo26s_img640")
csv_path = run_dir / "results.csv"

df = pd.read_csv(csv_path)
df.columns = [c.strip() for c in df.columns]

print("Columns:")
print(df.columns.tolist())

metric_col = "metrics/mAP50-95(B)"
best_idx = df[metric_col].idxmax()
best_row = df.loc[best_idx]

print("\nBEST BY mAP50-95:")
print("epoch:", int(best_row["epoch"]))
print("precision:", float(best_row["metrics/precision(B)"]))
print("recall:", float(best_row["metrics/recall(B)"]))
print("mAP50:", float(best_row["metrics/mAP50(B)"]))
print("mAP50-95:", float(best_row["metrics/mAP50-95(B)"]))

print("\nLAST EPOCH:")
last = df.iloc[-1]
print("epoch:", int(last["epoch"]))
print("precision:", float(last["metrics/precision(B)"]))
print("recall:", float(last["metrics/recall(B)"]))
print("mAP50:", float(last["metrics/mAP50(B)"]))
print("mAP50-95:", float(last["metrics/mAP50-95(B)"]))

print("\nWEIGHTS:")
print("best:", run_dir / "weights" / "best.pt")
print("last:", run_dir / "weights" / "last.pt")

Columns:
['epoch', 'time', 'train/box_loss', 'train/cls_loss', 'train/dfl_loss', 'metrics/precision(B)', 'metrics/recall(B)', 'metrics/mAP50(B)', 'metrics/mAP50-95(B)', 'val/box_loss', 'val/cls_loss', 'val/dfl_loss', 'lr/pg0', 'lr/pg1', 'lr/pg2']

BEST BY mAP50-95:
epoch: 71
precision: 0.82493
recall: 0.66468
mAP50: 0.76095
mAP50-95: 0.54024

LAST EPOCH:
epoch: 100
precision: 0.80521
recall: 0.64634
mAP50: 0.72757
mAP50-95: 0.51764

WEIGHTS:
best: /workspace/uavape_v4_model_benchmark/runs/v4_yolo26s_img640/weights/best.pt
last: /workspace/uavape_v4_model_benchmark/runs/v4_yolo26s_img640/weights/last.pt


In [16]:
!pip install -q pandas

In [18]:
from ultralytics import YOLO
from pathlib import Path

PROJECT = Path("/workspace/uavape_v4_model_benchmark")
RUN = {"model": "yolo26s.pt", "imgsz": 960}

model = YOLO(RUN["model"])

results = model.train(
    data=str(yaml_path),
    project=str(PROJECT / "runs"),
    name=f'v4_{Path(RUN["model"]).stem}_img{RUN["imgsz"]}',
    imgsz=RUN["imgsz"],
    epochs=100,
    batch=32,
    device=0,
    workers=8,
    seed=42,
    deterministic=True,
    cos_lr=True,
    patience=100,
    exist_ok=True,
    plots=True,
)

New https://pypi.org/project/ultralytics/8.4.63 available 😃 Update with 'pip install -U ultralytics'
Ultralytics 8.4.62 🚀 Python-3.12.3 torch-2.8.0+cu128 CUDA:0 (NVIDIA H100 80GB HBM3, 81079MiB)
engine/trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=32, bgr=0.0, box=7.5, cache=False, cfg=None, classes=None, close_mosaic=10, cls=0.5, cls_pw=0.0, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=True, cutmix=0.0, data=/workspace/uavape_v4_model_benchmark/datasets/v4_real_final_20260610/dataset.yaml, degrees=0.0, deterministic=True, device=0, dfl=1.5, dnn=False, dropout=0.0, dynamic=False, embed=None, end2end=None, epochs=100, erasing=0.4, exist_ok=True, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, half=False, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=960, int8=False, iou=0.7, keras=False, kobj=1.0, line_width=None, lr0=0.01, lrf=0.01, mask_ratio=4, max_det=300, mixup=0.0, mode=train, model=yolo

In [19]:
from ultralytics import YOLO
from pathlib import Path

PROJECT = Path("/workspace/uavape_v4_model_benchmark")
RUN = {"model": "yolo26s.pt", "imgsz": 1280}

model = YOLO(RUN["model"])

results = model.train(
    data=str(yaml_path),
    project=str(PROJECT / "runs"),
    name=f'v4_{Path(RUN["model"]).stem}_img{RUN["imgsz"]}',
    imgsz=RUN["imgsz"],
    epochs=100,
    batch=16,
    device=0,
    workers=8,
    seed=42,
    deterministic=True,
    cos_lr=True,
    patience=100,
    exist_ok=True,
    plots=True,
)

New https://pypi.org/project/ultralytics/8.4.63 available 😃 Update with 'pip install -U ultralytics'
Ultralytics 8.4.62 🚀 Python-3.12.3 torch-2.8.0+cu128 CUDA:0 (NVIDIA H100 80GB HBM3, 81079MiB)
engine/trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=16, bgr=0.0, box=7.5, cache=False, cfg=None, classes=None, close_mosaic=10, cls=0.5, cls_pw=0.0, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=True, cutmix=0.0, data=/workspace/uavape_v4_model_benchmark/datasets/v4_real_final_20260610/dataset.yaml, degrees=0.0, deterministic=True, device=0, dfl=1.5, dnn=False, dropout=0.0, dynamic=False, embed=None, end2end=None, epochs=100, erasing=0.4, exist_ok=True, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, half=False, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=1280, int8=False, iou=0.7, keras=False, kobj=1.0, line_width=None, lr0=0.01, lrf=0.01, mask_ratio=4, max_det=300, mixup=0.0, mode=train, model=yol

In [20]:
from ultralytics import YOLO
from pathlib import Path

PROJECT = Path("/workspace/uavape_v4_model_benchmark")

runs = [
    {"name": "v4_yolo26s_img640", "imgsz": 640},
    {"name": "v4_yolo26s_img960", "imgsz": 960},
    {"name": "v4_yolo26s_img1280", "imgsz": 1280},
]

for run in runs:
    run_dir = PROJECT / "runs" / run["name"]
    best_pt = run_dir / "weights" / "best.pt"

    print("\n" + "=" * 80)
    print(run["name"])
    print("=" * 80)

    model = YOLO(str(best_pt))

    metrics = model.val(
        data=str(yaml_path),
        split="val",
        imgsz=run["imgsz"],
        device=0,
        plots=False,
        verbose=False,
    )

    for class_id, class_name in metrics.names.items():
        p, r, ap50, ap5095 = metrics.box.class_result(class_id)
        print(
            f"{class_id} {class_name}: "
            f"P={p:.5f}, R={r:.5f}, AP50={ap50:.5f}, AP50-95={ap5095:.5f}"
        )

    print(
        "aggregate:",
        f"P={metrics.box.mp:.5f},",
        f"R={metrics.box.mr:.5f},",
        f"mAP50={metrics.box.map50:.5f},",
        f"mAP50-95={metrics.box.map:.5f}",
    )


v4_yolo26s_img640
Ultralytics 8.4.62 🚀 Python-3.12.3 torch-2.8.0+cu128 CUDA:0 (NVIDIA H100 80GB HBM3, 81079MiB)
YOLO26s summary (fused): 122 layers, 9,465,954 parameters, 0 gradients, 20.5 GFLOPs
val: Fast image access ✅ (ping: 0.3±0.2 ms, read: 1718.3±627.8 MB/s, size: 5754.2 KB)

val: /workspace/uavape_v4_model_benchmark/datasets/v4_real_final_20260610/images/val/upload_image_20260603_195809_img_2331.jpg: 1 duplicate labels removed
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 12/12 1.7it/s 7.0s
                   all        183        302      0.812      0.669      0.762      0.542
Speed: 0.5ms preprocess, 2.8ms inference, 0.0ms loss, 0.1ms postprocess per image
0 vape: P=0.88375, R=0.73762, AP50=0.87651, AP50-95=0.64776
1 lighter: P=0.74003, R=0.60000, AP50=0.64725, AP50-95=0.43683
aggregate: P=0.81189, R=0.66881, mAP50=0.76188, mAP50-95=0.54229

v4_yolo26s_img960
Ultralytics 8.4.62 🚀 Python-3.12.3 torch-2.8.0+cu128 CUD

In [25]:
print(yaml_path)
print(yaml_path.exists())
print(yaml_path.read_text())

/workspace/uavape_v4_model_benchmark/datasets/v4_real_final_20260610/dataset.yaml
True
path: /workspace/uavape_v4_model_benchmark/datasets/v4_real_final_20260610
train: images/train
val: images/val
test: images/test
nc: 2
names:
  0: vape
  1: lighter



In [22]:
from pathlib import Path

for p in Path("/workspace").iterdir():
    if p.is_file():
        print(p.name, round(p.stat().st_size / 1024**3, 3), "GB")
    else:
        print(p.name, "DIR")

yolo11s.pt 0.018 GB
runs DIR
yolo26n.pt 0.005 GB
yolo26s.pt 0.019 GB
v4_real_final_20260610_yolo_export.tar.gz 3.054 GB
uavape_v4_model_benchmark DIR
.cache DIR
.ipynb_checkpoints DIR
Untitled.ipynb 0.0 GB


In [23]:
!rm /workspace/v4_real_final_20260610_yolo_export.tar.gz

In [24]:
from pathlib import Path

for p in Path("/workspace").iterdir():
    if p.is_file():
        print(p.name, round(p.stat().st_size / 1024**3, 3), "GB")
    else:
        print(p.name, "DIR")

v4_real_final_20260610_yolo_export.tar.gz 0.004 GB
yolo11s.pt 0.018 GB
runs DIR
yolo26n.pt 0.005 GB
yolo26s.pt 0.019 GB
uavape_v4_model_benchmark DIR
.cache DIR
.ipynb_checkpoints DIR
Untitled.ipynb 0.0 GB


In [26]:
from ultralytics import YOLO
from pathlib import Path
import gc
import torch

PROJECT = Path("/workspace/uavape_v4_model_benchmark")

RUN_QUEUE = [
    # Stable modern comparator, feasible sizes
    {"model": "yolo11s.pt", "imgsz": 640, "batch": 32},
    {"model": "yolo11s.pt", "imgsz": 960, "batch": 32},

    # Literature-anchor comparator, feasible sizes
    {"model": "yolov8s.pt", "imgsz": 640, "batch": 32},
    {"model": "yolov8s.pt", "imgsz": 960, "batch": 32},

    # Capacity check at deployment-feasible input
    {"model": "yolo26m.pt", "imgsz": 640, "batch": 32},
    {"model": "yolo11m.pt", "imgsz": 640, "batch": 32},
    {"model": "yolov8m.pt", "imgsz": 640, "batch": 32},

    # Upper-resolution checks last
    {"model": "yolo11s.pt", "imgsz": 1280, "batch": 16},
    {"model": "yolov8s.pt", "imgsz": 1280, "batch": 16},
]

completed = []

for run in RUN_QUEUE:
    model_name = run["model"]
    imgsz = run["imgsz"]
    batch = run["batch"]
    run_name = f"v4_{Path(model_name).stem}_img{imgsz}"
    run_dir = PROJECT / "runs" / run_name

    if (run_dir / "weights" / "best.pt").exists():
        print(f"Skipping existing run: {run_name}")
        completed.append(run_name)
        continue

    print("\n" + "=" * 80)
    print(f"Training {run_name} | batch={batch}")
    print("=" * 80)

    try:
        model = YOLO(model_name)
        model.train(
            data=str(yaml_path),
            project=str(PROJECT / "runs"),
            name=run_name,
            imgsz=imgsz,
            epochs=100,
            batch=batch,
            device=0,
            workers=8,
            seed=42,
            deterministic=True,
            cos_lr=True,
            patience=100,
            exist_ok=True,
            plots=True,
        )
        completed.append(run_name)

    except RuntimeError as e:
        print(f"Run failed: {run_name}")
        print(e)
        if "out of memory" in str(e).lower():
            print("OOM detected. Rerun manually with lower batch.")
        break

    finally:
        gc.collect()
        if torch.cuda.is_available():
            torch.cuda.empty_cache()

print("Completed:", completed)


Training v4_yolo11s_img640 | batch=32
New https://pypi.org/project/ultralytics/8.4.63 available 😃 Update with 'pip install -U ultralytics'
Ultralytics 8.4.62 🚀 Python-3.12.3 torch-2.8.0+cu128 CUDA:0 (NVIDIA H100 80GB HBM3, 81079MiB)
engine/trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=32, bgr=0.0, box=7.5, cache=False, cfg=None, classes=None, close_mosaic=10, cls=0.5, cls_pw=0.0, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=True, cutmix=0.0, data=/workspace/uavape_v4_model_benchmark/datasets/v4_real_final_20260610/dataset.yaml, degrees=0.0, deterministic=True, device=0, dfl=1.5, dnn=False, dropout=0.0, dynamic=False, embed=None, end2end=None, epochs=100, erasing=0.4, exist_ok=True, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, half=False, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=640, int8=False, iou=0.7, keras=False, kobj=1.0, line_width=None, lr0=0.01, lrf=0.01, mask_ratio=4, max_det

In [28]:
!find runs -path "*/weights/best.pt" -o -path "*/results.csv" -o -path "*/args.yaml" | sort

In [29]:
from pathlib import Path
import pandas as pd

for csv in sorted(Path("runs").rglob("results.csv")):
    run = csv.parent
    try:
        df = pd.read_csv(csv)
        df.columns = [c.strip() for c in df.columns]

        last_epoch = int(df["epoch"].iloc[-1])
        metric_cols = [c for c in df.columns if "metrics/" in c]

        candidates = [c for c in metric_cols if "mAP50-95" in c]
        best_idx = df[candidates[0]].idxmax() if candidates else len(df) - 1

        best = df.loc[best_idx]

        print("\n" + "="*80)
        print(run)
        print(f"epochs completed: {last_epoch + 1}")
        print(f"best epoch: {int(best['epoch'])}")

        for c in metric_cols:
            print(f"{c}: {best[c]:.5f}")

        print("best.pt:", (run / "weights" / "best.pt").exists())
        print("last.pt:", (run / "weights" / "last.pt").exists())

    except Exception as e:
        print("\nFAILED TO READ:", csv, e)

In [30]:
from pathlib import Path
import pandas as pd

# Try common run locations
roots = [
    Path("runs"),
    Path("/content/runs"),
    Path("/workspace/runs"),
    Path.cwd() / "runs",
]

existing_roots = [p for p in roots if p.exists()]
print("Current folder:", Path.cwd())
print("Existing run folders:", existing_roots)

if not existing_roots:
    print("No runs folder found. Show folders in current directory:")
    print([p.name for p in Path.cwd().iterdir() if p.is_dir()])

Current folder: /workspace
Existing run folders: [PosixPath('runs'), PosixPath('/workspace/runs'), PosixPath('/workspace/runs')]


In [31]:
from pathlib import Path
import pandas as pd

root = existing_roots[0] if existing_roots else Path.cwd()
csvs = sorted(root.rglob("results.csv"))

print(f"Found {len(csvs)} results.csv files under {root}")

for csv in csvs:
    run = csv.parent
    df = pd.read_csv(csv)
    df.columns = [c.strip() for c in df.columns]

    print("\n" + "=" * 90)
    print("RUN:", run)
    print("ROWS:", len(df))
    
    if "epoch" in df.columns:
        print("EPOCHS COMPLETED:", int(df["epoch"].iloc[-1]) + 1)

    print("HAS best.pt:", (run / "weights" / "best.pt").exists())
    print("HAS last.pt:", (run / "weights" / "last.pt").exists())

    metric_cols = [c for c in df.columns if c.startswith("metrics/")]
    print("METRIC COLUMNS:", metric_cols)

    map_cols = [c for c in metric_cols if "mAP50-95" in c]
    if map_cols:
        best_idx = df[map_cols[0]].idxmax()
    else:
        best_idx = len(df) - 1

    best = df.iloc[best_idx]
    print("BEST ROW INDEX:", best_idx)

    for c in metric_cols:
        try:
            print(f"{c}: {float(best[c]):.5f}")
        except:
            print(f"{c}: {best[c]}")

Found 0 results.csv files under runs


In [32]:
from pathlib import Path

for p in Path.cwd().rglob("*"):
    if p.name in ["results.csv", "best.pt", "last.pt", "args.yaml"]:
        print(p)

/workspace/uavape_v4_model_benchmark/runs/v4_yolov8s_img1280/results.csv
/workspace/uavape_v4_model_benchmark/runs/v4_yolov8s_img1280/args.yaml
/workspace/uavape_v4_model_benchmark/runs/v4_yolo11s_img1280/results.csv
/workspace/uavape_v4_model_benchmark/runs/v4_yolo11s_img1280/args.yaml
/workspace/uavape_v4_model_benchmark/runs/v4_yolov8m_img640/results.csv
/workspace/uavape_v4_model_benchmark/runs/v4_yolov8m_img640/args.yaml
/workspace/uavape_v4_model_benchmark/runs/v4_yolo11m_img640/results.csv
/workspace/uavape_v4_model_benchmark/runs/v4_yolo11m_img640/args.yaml
/workspace/uavape_v4_model_benchmark/runs/v4_yolo26m_img640/results.csv
/workspace/uavape_v4_model_benchmark/runs/v4_yolo26m_img640/args.yaml
/workspace/uavape_v4_model_benchmark/runs/v4_yolov8s_img960/results.csv
/workspace/uavape_v4_model_benchmark/runs/v4_yolov8s_img960/args.yaml
/workspace/uavape_v4_model_benchmark/runs/v4_yolov8s_img640/results.csv
/workspace/uavape_v4_model_benchmark/runs/v4_yolov8s_img640/args.yaml
/w

In [33]:
from pathlib import Path
import pandas as pd

root = Path("/workspace/uavape_v4_model_benchmark/runs")
csvs = sorted(root.rglob("results.csv"))

rows = []

for csv in csvs:
    run_dir = csv.parent
    name = run_dir.name

    df = pd.read_csv(csv)
    df.columns = [c.strip() for c in df.columns]

    # Find mAP50-95 column
    map95_cols = [c for c in df.columns if "mAP50-95" in c]
    map50_cols = [c for c in df.columns if "mAP50(B)" in c or "mAP50" in c and "95" not in c]
    precision_cols = [c for c in df.columns if "precision" in c]
    recall_cols = [c for c in df.columns if "recall" in c]

    if map95_cols:
        best_idx = df[map95_cols[0]].idxmax()
    else:
        best_idx = len(df) - 1

    best = df.loc[best_idx]

    rows.append({
        "run": name,
        "epochs_done": int(df["epoch"].iloc[-1]) + 1 if "epoch" in df.columns else len(df),
        "best_epoch": int(best["epoch"]) if "epoch" in df.columns else best_idx,
        "precision": float(best[precision_cols[0]]) if precision_cols else None,
        "recall": float(best[recall_cols[0]]) if recall_cols else None,
        "mAP50": float(best[map50_cols[0]]) if map50_cols else None,
        "mAP50-95": float(best[map95_cols[0]]) if map95_cols else None,
        "best_pt": (run_dir / "weights" / "best.pt").exists(),
        "last_pt": (run_dir / "weights" / "last.pt").exists(),
    })

summary = pd.DataFrame(rows).sort_values("mAP50-95", ascending=False)
summary

,run,epochs_done,best_epoch,precision,recall,mAP50,mAP50-95,best_pt,last_pt
5,v4_yolo26s_img1280,101,77,0.83866,0.74882,0.80233,0.60501,True,True
1,v4_yolo11s_img1280,101,74,0.83625,0.75079,0.78559,0.58851,True,True
3,v4_yolo11s_img960,101,89,0.82587,0.74317,0.77487,0.56639,True,True
4,v4_yolo26m_img640,101,82,0.77691,0.75584,0.79481,0.56215,True,True
7,v4_yolo26s_img960,101,90,0.82190,0.68432,0.74253,0.54957,True,True
9,v4_yolov8s_img1280,101,76,0.85542,0.66381,0.73735,0.54448,True,True
6,v4_yolo26s_img640,101,71,0.82493,0.66468,0.76095,0.54024,True,True
11,v4_yolov8s_img960,101,89,0.82325,0.67099,0.73797,0.53573,True,True
8,v4_yolov8m_img640,101,88,0.78257,0.71402,0.75316,0.53368,True,True
0,v4_yolo11m_img640,101,81,0.86303,0.68856,0.75816,0.52415,True,True


In [35]:
from pathlib import Path

root = Path("/workspace/uavape_v4_model_benchmark")

yamls = sorted(list(root.rglob("*.yaml")) + list(root.rglob("*.yml")))

for y in yamls:
    print(y)

/workspace/uavape_v4_model_benchmark/datasets/v4_real_final_20260610/dataset.yaml
/workspace/uavape_v4_model_benchmark/runs/v4_yolo11m_img640/args.yaml
/workspace/uavape_v4_model_benchmark/runs/v4_yolo11s_img1280/args.yaml
/workspace/uavape_v4_model_benchmark/runs/v4_yolo11s_img640/args.yaml
/workspace/uavape_v4_model_benchmark/runs/v4_yolo11s_img960/args.yaml
/workspace/uavape_v4_model_benchmark/runs/v4_yolo26m_img640/args.yaml
/workspace/uavape_v4_model_benchmark/runs/v4_yolo26s_img1280/args.yaml
/workspace/uavape_v4_model_benchmark/runs/v4_yolo26s_img640/args.yaml
/workspace/uavape_v4_model_benchmark/runs/v4_yolo26s_img960/args.yaml
/workspace/uavape_v4_model_benchmark/runs/v4_yolov8m_img640/args.yaml
/workspace/uavape_v4_model_benchmark/runs/v4_yolov8s_img1280/args.yaml
/workspace/uavape_v4_model_benchmark/runs/v4_yolov8s_img640/args.yaml
/workspace/uavape_v4_model_benchmark/runs/v4_yolov8s_img960/args.yaml


In [36]:
for y in yamls:
    txt = y.read_text(errors="ignore")
    if "train:" in txt and "val:" in txt and "names:" in txt:
        print("\n" + "="*80)
        print(y)
        print(txt[:1000])


/workspace/uavape_v4_model_benchmark/datasets/v4_real_final_20260610/dataset.yaml
path: /workspace/uavape_v4_model_benchmark/datasets/v4_real_final_20260610
train: images/train
val: images/val
test: images/test
nc: 2
names:
  0: vape
  1: lighter



In [37]:
data_yaml = "/workspace/uavape_v4_model_benchmark/datasets/v4_real_final_20260610/dataset.yaml"

In [38]:
from pathlib import Path
from ultralytics import YOLO
import pandas as pd

runs = Path("/workspace/uavape_v4_model_benchmark/runs")
data_yaml = "/workspace/uavape_v4_model_benchmark/datasets/v4_real_final_20260610/dataset.yaml"

rows = []

for best in sorted(runs.rglob("weights/best.pt")):
    run_name = best.parents[1].name
    print("Validating:", run_name)

    model = YOLO(str(best))
    metrics = model.val(
        data=data_yaml,
        split="val",
        plots=False,
        save_json=False,
        verbose=False
    )

    names = metrics.names
    ap50 = metrics.box.ap50
    ap = metrics.box.ap
    p = metrics.box.p
    r = metrics.box.r

    for i, cls_name in names.items():
        rows.append({
            "run": run_name,
            "class": cls_name,
            "precision": float(p[i]),
            "recall": float(r[i]),
            "AP50": float(ap50[i]),
            "AP50-95": float(ap[i]),
        })

per_class = pd.DataFrame(rows)
per_class.to_csv("/workspace/uavape_v4_model_benchmark/v4_per_class_val_summary.csv", index=False)

per_class.sort_values(["class", "AP50-95"], ascending=[True, False])

Validating: v4_yolo11m_img640
Ultralytics 8.4.62 🚀 Python-3.12.3 torch-2.8.0+cu128 CUDA:0 (NVIDIA H100 80GB HBM3, 81079MiB)
YOLO11m summary (fused): 126 layers, 20,031,574 parameters, 0 gradients, 67.7 GFLOPs
val: Fast image access ✅ (ping: 0.9±0.3 ms, read: 189.0±73.1 MB/s, size: 4319.3 KB)

val: /workspace/uavape_v4_model_benchmark/datasets/v4_real_final_20260610/images/val/upload_image_20260603_195809_img_2331.jpg: 1 duplicate labels removed
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 12/12 1.3it/s 9.1s
                   all        183        302      0.867      0.686      0.759      0.526
Speed: 0.5ms preprocess, 3.3ms inference, 0.0ms loss, 0.5ms postprocess per image
Validating: v4_yolo11s_img1280
Ultralytics 8.4.62 🚀 Python-3.12.3 torch-2.8.0+cu128 CUDA:0 (NVIDIA H100 80GB HBM3, 81079MiB)
YOLO11s summary (fused): 101 layers, 9,413,574 parameters, 0 gradients, 21.3 GFLOPs
val: Fast image access ✅ (ping: 1.0±0.3 ms, 

,run,class,precision,recall,AP50,AP50-95
3,v4_yolo11s_img1280,lighter,0.784678,0.660000,0.690710,0.508317
11,v4_yolo26s_img1280,lighter,0.769843,0.660000,0.687925,0.497680
19,v4_yolov8s_img1280,lighter,0.865730,0.590000,0.671865,0.476142
7,v4_yolo11s_img960,lighter,0.776248,0.630000,0.647348,0.457565
9,v4_yolo26m_img640,lighter,0.697677,0.690000,0.706005,0.455211
13,v4_yolo26s_img640,lighter,0.740030,0.600000,0.647245,0.436827
17,v4_yolov8m_img640,lighter,0.696642,0.620038,0.638209,0.427761
23,v4_yolov8s_img960,lighter,0.770661,0.537682,0.606621,0.418280
15,v4_yolo26s_img960,lighter,0.755171,0.620000,0.622254,0.411715
1,v4_yolo11m_img640,lighter,0.855696,0.590000,0.635734,0.403049


In [39]:
from ultralytics import YOLO
from pathlib import Path

data_yaml = "/workspace/uavape_v4_model_benchmark/datasets/v4_real_final_20260610/dataset.yaml"
project = "/workspace/uavape_v4_model_benchmark/runs"

runs = [
    {
        "model": "yolo26n.pt",
        "imgsz": 640,
        "name": "v4_yolo26n_img640",
    },
    {
        "model": "yolo26n.pt",
        "imgsz": 960,
        "name": "v4_yolo26n_img960",
    },
    {
        "model": "yolo11n.pt",
        "imgsz": 640,
        "name": "v4_yolo11n_img640",
    },
    {
        "model": "yolo11n.pt",
        "imgsz": 960,
        "name": "v4_yolo11n_img960",
    },
]

for r in runs:
    out_dir = Path(project) / r["name"]

    if (out_dir / "weights" / "best.pt").exists():
        print(f"Skipping completed run: {r['name']}")
        continue

    print(f"\nStarting {r['name']}")

    model = YOLO(r["model"])
    model.train(
        data=data_yaml,
        epochs=100,
        imgsz=r["imgsz"],
        seed=42,
        project=project,
        name=r["name"],
        exist_ok=True,
        cos_lr=True,
        plots=True,
        verbose=True,
    )


Starting v4_yolo26n_img640
New https://pypi.org/project/ultralytics/8.4.63 available 😃 Update with 'pip install -U ultralytics'
Ultralytics 8.4.62 🚀 Python-3.12.3 torch-2.8.0+cu128 CUDA:0 (NVIDIA H100 80GB HBM3, 81079MiB)
engine/trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=16, bgr=0.0, box=7.5, cache=False, cfg=None, classes=None, close_mosaic=10, cls=0.5, cls_pw=0.0, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=True, cutmix=0.0, data=/workspace/uavape_v4_model_benchmark/datasets/v4_real_final_20260610/dataset.yaml, degrees=0.0, deterministic=True, device=0, dfl=1.5, dnn=False, dropout=0.0, dynamic=False, embed=None, end2end=None, epochs=100, erasing=0.4, exist_ok=True, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, half=False, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=640, int8=False, iou=0.7, keras=False, kobj=1.0, line_width=None, lr0=0.01, lrf=0.01, mask_ratio=4, max_det=300, mixup

In [40]:
!pip install -q sahi pycocotools pyyaml tqdm

In [41]:
from pathlib import Path
from PIL import Image
from tqdm import tqdm
import json
import csv
import yaml
import contextlib
import io

from sahi import AutoDetectionModel
from sahi.predict import get_sliced_prediction
from pycocotools.coco import COCO
from pycocotools.cocoeval import COCOeval

PROJECT = Path("/workspace/uavape_v4_model_benchmark")
DATA_DIR = PROJECT / "datasets" / "v4_real_final_20260610"
VAL_IMG_DIR = DATA_DIR / "images" / "val"
VAL_LBL_DIR = DATA_DIR / "labels" / "val"
OUT_DIR = PROJECT / "tiling_eval"
OUT_DIR.mkdir(parents=True, exist_ok=True)

CATEGORIES = [
    {"id": 1, "name": "vape"},
    {"id": 2, "name": "lighter"},
]

TILE_RUNS = [
    {
        "name": "v4_yolo26s_img640_sahi_tile1280_ov20",
        "weights": PROJECT / "runs" / "v4_yolo26s_img640" / "weights" / "best.pt",
        "model_imgsz": 640,
        "slice_size": 1280,
        "overlap": 0.20,
    },
    {
        "name": "v4_yolo26s_img960_sahi_tile1280_ov20",
        "weights": PROJECT / "runs" / "v4_yolo26s_img960" / "weights" / "best.pt",
        "model_imgsz": 960,
        "slice_size": 1280,
        "overlap": 0.20,
    },
    {
        "name": "v4_yolo11s_img960_sahi_tile1280_ov20",
        "weights": PROJECT / "runs" / "v4_yolo11s_img960" / "weights" / "best.pt",
        "model_imgsz": 960,
        "slice_size": 1280,
        "overlap": 0.20,
    },
    # Optional deployment-boundary check
    {
        "name": "v4_yolo11n_img960_sahi_tile1280_ov20",
        "weights": PROJECT / "runs" / "v4_yolo11n_img960" / "weights" / "best.pt",
        "model_imgsz": 960,
        "slice_size": 1280,
        "overlap": 0.20,
    },
]

def yolo_to_coco_gt():
    images = []
    annotations = []
    ann_id = 1

    image_paths = sorted(
        p for p in VAL_IMG_DIR.iterdir()
        if p.suffix.lower() in [".jpg", ".jpeg", ".png"]
        and not p.name.startswith("._")
    )

    for img_id, img_path in enumerate(image_paths, start=1):
        with Image.open(img_path) as im:
            w, h = im.size

        images.append({
            "id": img_id,
            "file_name": img_path.name,
            "width": w,
            "height": h,
        })

        label_path = VAL_LBL_DIR / f"{img_path.stem}.txt"
        if not label_path.exists():
            continue

        seen = set()
        for line in label_path.read_text(errors="ignore").splitlines():
            parts = line.strip().split()
            if len(parts) != 5:
                continue

            cls, xc, yc, bw, bh = map(float, parts)
            cls = int(cls)

            # mirror Ultralytics duplicate cleanup
            key = tuple(round(x, 8) for x in [cls, xc, yc, bw, bh])
            if key in seen:
                continue
            seen.add(key)

            x = (xc - bw / 2) * w
            y = (yc - bh / 2) * h
            box_w = bw * w
            box_h = bh * h

            annotations.append({
                "id": ann_id,
                "image_id": img_id,
                "category_id": cls + 1,
                "bbox": [x, y, box_w, box_h],
                "area": box_w * box_h,
                "iscrowd": 0,
            })
            ann_id += 1

    return {
        "images": images,
        "annotations": annotations,
        "categories": CATEGORIES,
    }

GT_JSON = OUT_DIR / "val_gt_coco.json"
if not GT_JSON.exists():
    gt = yolo_to_coco_gt()
    GT_JSON.write_text(json.dumps(gt))
    print(f"Wrote GT: {GT_JSON} | images={len(gt['images'])}, anns={len(gt['annotations'])}")
else:
    print(f"Using existing GT: {GT_JSON}")

gt_data = json.loads(GT_JSON.read_text())
image_id_by_name = {img["file_name"]: img["id"] for img in gt_data["images"]}
image_paths = [VAL_IMG_DIR / img["file_name"] for img in gt_data["images"]]

def build_sahi_model(weights, model_imgsz, conf=0.001):
    # Try newer SAHI model type first, then fallback used by older SAHI builds.
    last_error = None
    for model_type in ["ultralytics", "yolov8"]:
        try:
            return AutoDetectionModel.from_pretrained(
                model_type=model_type,
                model_path=str(weights),
                confidence_threshold=conf,
                device="cuda:0",
                image_size=model_imgsz,
            )
        except Exception as e:
            last_error = e
    raise last_error

def run_sahi_predictions(run):
    run_out = OUT_DIR / run["name"]
    run_out.mkdir(parents=True, exist_ok=True)

    pred_json = run_out / "predictions.json"
    if pred_json.exists():
        print(f"Using existing predictions: {pred_json}")
        return pred_json

    assert run["weights"].exists(), f"Missing weights: {run['weights']}"

    model = build_sahi_model(run["weights"], run["model_imgsz"])
    preds = []

    for img_path in tqdm(image_paths, desc=run["name"]):
        result = get_sliced_prediction(
            image=str(img_path),
            detection_model=model,
            slice_height=run["slice_size"],
            slice_width=run["slice_size"],
            overlap_height_ratio=run["overlap"],
            overlap_width_ratio=run["overlap"],
            postprocess_type="NMS",
            postprocess_match_metric="IOU",
            postprocess_match_threshold=0.5,
            verbose=0,
        )

        image_id = image_id_by_name[img_path.name]

        for obj in result.object_prediction_list:
            cls_id = int(obj.category.id)
            score = float(obj.score.value)

            x1, y1, x2, y2 = obj.bbox.to_xyxy()
            x = float(x1)
            y = float(y1)
            w = float(x2 - x1)
            h = float(y2 - y1)

            if w <= 0 or h <= 0:
                continue

            preds.append({
                "image_id": image_id,
                "category_id": cls_id + 1,
                "bbox": [x, y, w, h],
                "score": score,
            })

    pred_json.write_text(json.dumps(preds))
    print(f"Wrote predictions: {pred_json} | preds={len(preds)}")
    return pred_json

def coco_eval_metrics(pred_json):
    coco_gt = COCO(str(GT_JSON))

    preds = json.loads(Path(pred_json).read_text())
    if len(preds) == 0:
        return {
            "aggregate_AP50": 0.0,
            "aggregate_AP50_95": 0.0,
            "vape_AP50": 0.0,
            "vape_AP50_95": 0.0,
            "lighter_AP50": 0.0,
            "lighter_AP50_95": 0.0,
        }

    coco_dt = coco_gt.loadRes(str(pred_json))

    def run_eval(cat_ids=None):
        ev = COCOeval(coco_gt, coco_dt, "bbox")
        if cat_ids is not None:
            ev.params.catIds = cat_ids
        ev.params.imgIds = sorted(image_id_by_name.values())

        # quiet pycocotools chatter
        with contextlib.redirect_stdout(io.StringIO()):
            ev.evaluate()
            ev.accumulate()
            ev.summarize()

        return {
            "AP50_95": float(ev.stats[0]),
            "AP50": float(ev.stats[1]),
        }

    agg = run_eval()
    vape = run_eval([1])
    lighter = run_eval([2])

    return {
        "aggregate_AP50": agg["AP50"],
        "aggregate_AP50_95": agg["AP50_95"],
        "vape_AP50": vape["AP50"],
        "vape_AP50_95": vape["AP50_95"],
        "lighter_AP50": lighter["AP50"],
        "lighter_AP50_95": lighter["AP50_95"],
    }

summary_rows = []

for run in TILE_RUNS:
    print("\n" + "=" * 80)
    print(run["name"])
    print("=" * 80)

    pred_json = run_sahi_predictions(run)
    metrics = coco_eval_metrics(pred_json)

    row = {
        "run": run["name"],
        "weights": str(run["weights"]),
        "model_imgsz": run["model_imgsz"],
        "slice_size": run["slice_size"],
        "overlap": run["overlap"],
        **metrics,
    }
    summary_rows.append(row)

    print(row)

summary_csv = OUT_DIR / "sahi_tiling_validation_summary.csv"
with summary_csv.open("w", newline="") as f:
    writer = csv.DictWriter(f, fieldnames=list(summary_rows[0].keys()))
    writer.writeheader()
    writer.writerows(summary_rows)

print("\nSaved:", summary_csv)
print("\nSummary:")
for row in summary_rows:
    print(row)

Wrote GT: /workspace/uavape_v4_model_benchmark/tiling_eval/val_gt_coco.json | images=183, anns=302

v4_yolo26s_img640_sahi_tile1280_ov20


v4_yolo26s_img640_sahi_tile1280_ov20: 100%|██████████| 183/183 [02:03<00:00,  1.48it/s]


Wrote predictions: /workspace/uavape_v4_model_benchmark/tiling_eval/v4_yolo26s_img640_sahi_tile1280_ov20/predictions.json | preds=9131
loading annotations into memory...
Done (t=0.00s)
creating index...
index created!
Loading and preparing results...
DONE (t=0.03s)
creating index...
index created!
{'run': 'v4_yolo26s_img640_sahi_tile1280_ov20', 'weights': '/workspace/uavape_v4_model_benchmark/runs/v4_yolo26s_img640/weights/best.pt', 'model_imgsz': 640, 'slice_size': 1280, 'overlap': 0.2, 'aggregate_AP50': 0.6398191379754871, 'aggregate_AP50_95': 0.484360499916287, 'vape_AP50': 0.7274123196699179, 'vape_AP50_95': 0.55774439313922, 'lighter_AP50': 0.5522259562810561, 'lighter_AP50_95': 0.410976606693354}

v4_yolo26s_img960_sahi_tile1280_ov20


v4_yolo26s_img960_sahi_tile1280_ov20:  57%|█████▋    | 104/183 [00:40<00:14,  5.33it/s]2026-06-10 15:27:27,162 - sahi - WARNING - ignoring invalid prediction with bbox: [1536, 0, 1536, 156.87229919433594] (ultralytics.py:285)
2026-06-10 15:27:27,163 - sahi - WARNING - ignoring invalid prediction with bbox: [1536, 0, 1536, 177.50228881835938] (ultralytics.py:285)
v4_yolo26s_img960_sahi_tile1280_ov20: 100%|██████████| 183/183 [02:06<00:00,  1.44it/s]


Wrote predictions: /workspace/uavape_v4_model_benchmark/tiling_eval/v4_yolo26s_img960_sahi_tile1280_ov20/predictions.json | preds=7568
loading annotations into memory...
Done (t=0.00s)
creating index...
index created!
Loading and preparing results...
DONE (t=0.02s)
creating index...
index created!
{'run': 'v4_yolo26s_img960_sahi_tile1280_ov20', 'weights': '/workspace/uavape_v4_model_benchmark/runs/v4_yolo26s_img960/weights/best.pt', 'model_imgsz': 960, 'slice_size': 1280, 'overlap': 0.2, 'aggregate_AP50': 0.6959228035897583, 'aggregate_AP50_95': 0.5102650897720695, 'vape_AP50': 0.8055384418215006, 'vape_AP50_95': 0.6055932807417023, 'lighter_AP50': 0.5863071653580161, 'lighter_AP50_95': 0.41493689880243684}

v4_yolo11s_img960_sahi_tile1280_ov20


v4_yolo11s_img960_sahi_tile1280_ov20:  11%|█▏        | 21/183 [00:23<01:15,  2.14it/s]2026-06-10 15:29:17,319 - sahi - WARNING - ignoring invalid prediction with bbox: [1536, 1086.67041015625, 1536, 1310.8988037109375] (ultralytics.py:285)
2026-06-10 15:29:17,320 - sahi - WARNING - ignoring invalid prediction with bbox: [1536, 942.1649169921875, 1536, 1170.073486328125] (ultralytics.py:285)
v4_yolo11s_img960_sahi_tile1280_ov20:  32%|███▏      | 59/183 [00:30<00:21,  5.65it/s]2026-06-10 15:29:23,927 - sahi - WARNING - ignoring invalid prediction with bbox: [1536, 1892.464111328125, 1536, 2046.779296875] (ultralytics.py:285)
2026-06-10 15:29:23,927 - sahi - WARNING - ignoring invalid prediction with bbox: [1536, 1953.6737060546875, 1536, 2047.76171875] (ultralytics.py:285)
2026-06-10 15:29:23,928 - sahi - WARNING - ignoring invalid prediction with bbox: [1536, 1821.0570068359375, 1536, 2018.7994384765625] (ultralytics.py:285)
2026-06-10 15:29:23,928 - sahi - WARNING - ignoring invalid pr

Wrote predictions: /workspace/uavape_v4_model_benchmark/tiling_eval/v4_yolo11s_img960_sahi_tile1280_ov20/predictions.json | preds=10449
loading annotations into memory...
Done (t=0.00s)
creating index...
index created!
Loading and preparing results...
DONE (t=0.28s)
creating index...
index created!
{'run': 'v4_yolo11s_img960_sahi_tile1280_ov20', 'weights': '/workspace/uavape_v4_model_benchmark/runs/v4_yolo11s_img960/weights/best.pt', 'model_imgsz': 960, 'slice_size': 1280, 'overlap': 0.2, 'aggregate_AP50': 0.680782721685517, 'aggregate_AP50_95': 0.5112363991427478, 'vape_AP50': 0.7659535711757335, 'vape_AP50_95': 0.5847864098080153, 'lighter_AP50': 0.5956118721953008, 'lighter_AP50_95': 0.43768638847748004}

v4_yolo11n_img960_sahi_tile1280_ov20


v4_yolo11n_img960_sahi_tile1280_ov20: 100%|██████████| 183/183 [02:02<00:00,  1.50it/s]


Wrote predictions: /workspace/uavape_v4_model_benchmark/tiling_eval/v4_yolo11n_img960_sahi_tile1280_ov20/predictions.json | preds=14250
loading annotations into memory...
Done (t=0.00s)
creating index...
index created!
Loading and preparing results...
DONE (t=0.26s)
creating index...
index created!
{'run': 'v4_yolo11n_img960_sahi_tile1280_ov20', 'weights': '/workspace/uavape_v4_model_benchmark/runs/v4_yolo11n_img960/weights/best.pt', 'model_imgsz': 960, 'slice_size': 1280, 'overlap': 0.2, 'aggregate_AP50': 0.6780384370508052, 'aggregate_AP50_95': 0.5077306319346564, 'vape_AP50': 0.7587143200522484, 'vape_AP50_95': 0.5693738484529441, 'lighter_AP50': 0.5973625540493619, 'lighter_AP50_95': 0.4460874154163686}

Saved: /workspace/uavape_v4_model_benchmark/tiling_eval/sahi_tiling_validation_summary.csv

Summary:
{'run': 'v4_yolo26s_img640_sahi_tile1280_ov20', 'weights': '/workspace/uavape_v4_model_benchmark/runs/v4_yolo26s_img640/weights/best.pt', 'model_imgsz': 640, 'slice_size': 1280, 'ov

In [45]:
!pip install -q -U gdown

In [47]:
from pathlib import Path
import zipfile, shutil, re
import gdown

ROOT = Path("/workspace/uavape_v4_model_benchmark")
ADDONS = ROOT / "addons"
ADDONS.mkdir(parents=True, exist_ok=True)

DRIVE_FILES = {
    "scraped": "https://drive.google.com/file/d/1oApe7tstJrBlT1dulzC5wMw_ej-se6qg/view?usp=sharing",
    "synthetic": "https://drive.google.com/file/d/1kl0MgZg7iMzZKK81zo93pcTZYzFrj3kG/view?usp=sharing",
}

def drive_id_from_ref(ref):
    ref = str(ref).strip()
    if "drive.google.com" not in ref:
        return ref
    m = re.search(r"/file/d/([^/]+)", ref)
    if m:
        return m.group(1)
    m = re.search(r"[?&]id=([^&]+)", ref)
    if m:
        return m.group(1)
    raise ValueError(f"Could not extract Drive file ID from: {ref}")

def download_drive_file(name, file_ref):
    out_zip = ADDONS / f"{name}.zip"
    file_id = drive_id_from_ref(file_ref)
    url = f"https://drive.google.com/uc?id={file_id}"

    print(f"Downloading {name} from file id {file_id}")
    gdown.download(url, str(out_zip), quiet=False)

    if not out_zip.exists() or out_zip.stat().st_size == 0:
        raise FileNotFoundError(f"Download failed or empty: {out_zip}")
    print(name, "zip size MB:", round(out_zip.stat().st_size / 1e6, 2))
    return out_zip

def extract_source_zip(source_name, zip_path):
    out_dir = ADDONS / source_name
    if out_dir.exists():
        shutil.rmtree(out_dir)
    out_dir.mkdir(parents=True, exist_ok=True)

    with zipfile.ZipFile(zip_path, "r") as z:
        z.extractall(out_dir)

    image_dirs = list(out_dir.rglob("images"))
    label_dirs = list(out_dir.rglob("labels"))

    if image_dirs and label_dirs and image_dirs[0].parent != out_dir:
        nested = image_dirs[0].parent
        flat_tmp = ADDONS / f"_{source_name}_flat_tmp"
        if flat_tmp.exists():
            shutil.rmtree(flat_tmp)
        flat_tmp.mkdir(parents=True)
        shutil.copytree(nested / "images", flat_tmp / "images")
        shutil.copytree(nested / "labels", flat_tmp / "labels")
        shutil.rmtree(out_dir)
        flat_tmp.rename(out_dir)

    print(source_name, "ready:", out_dir)
    print("images:", len(list((out_dir / "images").glob("*"))))
    print("labels:", len(list((out_dir / "labels").glob("*.txt"))))

for name, file_ref in DRIVE_FILES.items():
    zip_path = download_drive_file(name, file_ref)
    extract_source_zip(name, zip_path)

Downloading...
From (original): https://drive.google.com/uc?id=1oApe7tstJrBlT1dulzC5wMw_ej-se6qg
From (redirected): https://drive.google.com/uc?id=1oApe7tstJrBlT1dulzC5wMw_ej-se6qg&confirm=t&uuid=2fe72a6e-9a89-4519-a68d-77757a90bf90
To: /workspace/uavape_v4_model_benchmark/addons/scraped.zip
100%|██████████| 271M/271M [00:04<00:00, 63.7MB/s] 


scraped zip size MB: 270.53
scraped ready: /workspace/uavape_v4_model_benchmark/addons/scraped
images: 2906
labels: 2906


Downloading...
From (original): https://drive.google.com/uc?id=1kl0MgZg7iMzZKK81zo93pcTZYzFrj3kG
From (redirected): https://drive.google.com/uc?id=1kl0MgZg7iMzZKK81zo93pcTZYzFrj3kG&confirm=t&uuid=140a61f6-ce50-4514-a062-c57cae283561
To: /workspace/uavape_v4_model_benchmark/addons/synthetic.zip
100%|██████████| 1.31G/1.31G [00:16<00:00, 81.1MB/s]


synthetic zip size MB: 1310.05
synthetic ready: /workspace/uavape_v4_model_benchmark/addons/synthetic
images: 355
labels: 355


In [48]:
from pathlib import Path
import random
import shutil
import csv
import yaml
from collections import Counter

ROOT = Path("/workspace/uavape_v4_model_benchmark")
BASE = ROOT / "datasets" / "v4_real_final_20260610"
OUT_ROOT = ROOT / "datasets" / "v4_data_ablation_20260610"

SEED = 42
DOSES = [88, 177, 355]
IMG_EXTS = {".jpg", ".jpeg", ".png", ".webp", ".bmp"}

SCRAPED_CANDIDATES = [
    ROOT / "addons" / "scraped",
    ROOT / "old_sources" / "external_vape_aggregated_vape_only" / "yolo_pool" / "external_positive",
    ROOT / "experiments" / "ml" / "results" / "external_vape_aggregated_vape_only" / "yolo_pool" / "external_positive",
]

SYNTHETIC_CANDIDATES = [
    ROOT / "addons" / "synthetic",
    ROOT / "old_sources" / "synthetic_promoted_only" / "splits" / "default_70_20_10_e3" / "train",
    ROOT / "experiments" / "ml" / "results" / "synthetic_promoted_only" / "splits" / "default_70_20_10_e3" / "train",
]

def normalise_pool_dir(pool_dir):
    pool_dir = Path(pool_dir)
    if (pool_dir / "images").exists() and (pool_dir / "labels").exists():
        return pool_dir / "images", pool_dir / "labels"
    if (pool_dir / "train" / "images").exists() and (pool_dir / "train" / "labels").exists():
        return pool_dir / "train" / "images", pool_dir / "train" / "labels"
    return None, None

def list_yolo_pairs(pool_dir):
    images_dir, labels_dir = normalise_pool_dir(pool_dir)
    if images_dir is None:
        return []
    pairs = []
    for img in sorted(images_dir.iterdir()):
        if img.suffix.lower() not in IMG_EXTS:
            continue
        lab = labels_dir / f"{img.stem}.txt"
        if lab.exists():
            pairs.append((img, lab))
    return pairs

def find_first_pool(candidates, label):
    checked = []
    for candidate in candidates:
        pairs = list_yolo_pairs(candidate)
        checked.append((candidate, len(pairs)))
        if pairs:
            print(f"{label} pool selected: {candidate} | labelled images={len(pairs)}")
            return Path(candidate)
    print(f"No {label} pool found. Checked:")
    for path, count in checked:
        print(f"  {path} | labelled images={count}")
    return None

def audit_pool(pool_dir, label):
    pairs = list_yolo_pairs(pool_dir)
    cls_counts = Counter()
    malformed = []
    empty = 0
    for _, lab in pairs:
        text = lab.read_text().splitlines()
        if not text:
            empty += 1
            continue
        for line in text:
            parts = line.split()
            if len(parts) != 5:
                malformed.append(str(lab))
                continue
            cls_counts[parts[0]] += 1
    print(f"\n{label} audit")
    print("pool:", pool_dir)
    print("labelled images:", len(pairs))
    print("class ids:", dict(sorted(cls_counts.items())))
    print("empty labels:", empty)
    print("malformed labels:", len(set(malformed)))
    if set(cls_counts.keys()) - {"0"}:
        raise ValueError(f"{label} contains class IDs other than 0. Do not use until remapped/audited.")
    if len(pairs) < max(DOSES):
        raise ValueError(f"{label} has only {len(pairs)} labelled images; need at least {max(DOSES)}.")
    return pairs

SCRAPED = find_first_pool(SCRAPED_CANDIDATES, "scraped")
SYNTHETIC = find_first_pool(SYNTHETIC_CANDIDATES, "synthetic")

if SCRAPED is None:
    raise FileNotFoundError("Scraped pool not found. Copy it into /workspace/uavape_v4_model_benchmark/addons/scraped or edit SCRAPED_CANDIDATES.")
if SYNTHETIC is None:
    raise FileNotFoundError("Synthetic pool not found. Copy it into /workspace/uavape_v4_model_benchmark/addons/synthetic or edit SYNTHETIC_CANDIDATES.")

scraped_pairs = audit_pool(SCRAPED, "scraped")
synthetic_pairs = audit_pool(SYNTHETIC, "synthetic")

print("\nOK: add-on pools are one-class class-0 sources, compatible with V4 class 0 = vape.")

scraped pool selected: /workspace/uavape_v4_model_benchmark/addons/scraped | labelled images=2906
synthetic pool selected: /workspace/uavape_v4_model_benchmark/addons/synthetic | labelled images=355

scraped audit
pool: /workspace/uavape_v4_model_benchmark/addons/scraped
labelled images: 2906
class ids: {'0': 4438}
empty labels: 0
malformed labels: 0

synthetic audit
pool: /workspace/uavape_v4_model_benchmark/addons/synthetic
labelled images: 355
class ids: {'0': 355}
empty labels: 0
malformed labels: 0

OK: add-on pools are one-class class-0 sources, compatible with V4 class 0 = vape.


In [49]:
from pathlib import Path
import random
import shutil
import csv
import yaml
from collections import Counter

def sample_nested_pairs(pool_dir, doses, seed=SEED):
    pairs = list_yolo_pairs(pool_dir)
    rng = random.Random(seed)
    shuffled = pairs[:]
    rng.shuffle(shuffled)
    max_n = max(doses)
    selected = shuffled[:max_n]
    return {n: sorted(selected[:n], key=lambda p: p[0].name) for n in doses}

def copy_split(src_dataset, dst_dataset, split):
    for kind in ["images", "labels"]:
        src = src_dataset / kind / split
        dst = dst_dataset / kind / split
        dst.mkdir(parents=True, exist_ok=True)
        for item in sorted(src.iterdir()):
            if item.is_file():
                shutil.copy2(item, dst / item.name)

def copy_addons(dst_dataset, pairs, prefix):
    img_dst = dst_dataset / "images" / "train"
    lab_dst = dst_dataset / "labels" / "train"
    rows = []
    for img, lab in pairs:
        img_name = f"{prefix}_{img.name}"
        lab_name = f"{prefix}_{lab.name}"
        shutil.copy2(img, img_dst / img_name)
        shutil.copy2(lab, lab_dst / lab_name)
        rows.append({"source": prefix, "image": str(img), "label": str(lab), "copied_image": img_name})
    return rows

def count_split(dataset, split):
    image_dir = dataset / "images" / split
    label_dir = dataset / "labels" / split
    image_count = len([p for p in image_dir.iterdir() if p.suffix.lower() in IMG_EXTS])
    label_count = len([p for p in label_dir.iterdir() if p.suffix.lower() == ".txt"])
    class_counts = Counter()
    box_count = 0
    for lab in label_dir.glob("*.txt"):
        for line in lab.read_text().splitlines():
            if not line.strip():
                continue
            cls = int(float(line.split()[0]))
            class_counts[cls] += 1
            box_count += 1
    return image_count, label_count, box_count, class_counts

def write_dataset_yaml(dataset):
    data = {
        "path": str(dataset),
        "train": "images/train",
        "val": "images/val",
        "test": "images/test",
        "names": {0: "vape", 1: "lighter"},
    }
    with open(dataset / "dataset.yaml", "w") as f:
        yaml.safe_dump(data, f, sort_keys=False)

def build_dataset(name, scraped=None, synthetic=None):
    scraped = scraped or []
    synthetic = synthetic or []
    dst = OUT_ROOT / name
    if dst.exists():
        shutil.rmtree(dst)
    for split in ["train", "val", "test"]:
        copy_split(BASE, dst, split)

    manifest_rows = []
    manifest_rows += copy_addons(dst, scraped, "scraped")
    manifest_rows += copy_addons(dst, synthetic, "synthetic")
    write_dataset_yaml(dst)

    with open(dst / "addon_manifest.csv", "w", newline="") as f:
        writer = csv.DictWriter(f, fieldnames=["source", "image", "label", "copied_image"])
        writer.writeheader()
        writer.writerows(manifest_rows)

    summary_rows = []
    for split in ["train", "val", "test"]:
        images, labels, boxes, class_counts = count_split(dst, split)
        summary_rows.append({
            "dataset": name,
            "split": split,
            "images": images,
            "labels": labels,
            "boxes": boxes,
            "vape_boxes": class_counts.get(0, 0),
            "lighter_boxes": class_counts.get(1, 0),
        })

    with open(dst / "dataset_summary.csv", "w", newline="") as f:
        writer = csv.DictWriter(f, fieldnames=summary_rows[0].keys())
        writer.writeheader()
        writer.writerows(summary_rows)

    print(f"\n{name}")
    for row in summary_rows:
        print(row)
    return dst

OUT_ROOT.mkdir(parents=True, exist_ok=True)

scraped_samples = sample_nested_pairs(SCRAPED, DOSES, seed=SEED)
synthetic_samples = sample_nested_pairs(SYNTHETIC, DOSES, seed=SEED)

ABLATION_DATASETS = []
ABLATION_DATASETS.append(build_dataset("v4_ablate_A0_real_only"))

for dose in DOSES:
    ABLATION_DATASETS.append(
        build_dataset(f"v4_ablate_S{dose}_scraped", scraped=scraped_samples[dose])
    )

for dose in DOSES:
    ABLATION_DATASETS.append(
        build_dataset(f"v4_ablate_Y{dose}_synthetic", synthetic=synthetic_samples[dose])
    )

ABLATION_DATASETS.append(
    build_dataset(
        "v4_ablate_S355_Y355_combined",
        scraped=scraped_samples[355],
        synthetic=synthetic_samples[355],
    )
)

ABLATIONS = [p.name for p in ABLATION_DATASETS]
print("\nAblations ready:")
for name in ABLATIONS:
    print(" ", name, OUT_ROOT / name / "dataset.yaml")


v4_ablate_A0_real_only
{'dataset': 'v4_ablate_A0_real_only', 'split': 'train', 'images': 837, 'labels': 837, 'boxes': 1317, 'vape_boxes': 870, 'lighter_boxes': 447}
{'dataset': 'v4_ablate_A0_real_only', 'split': 'val', 'images': 183, 'labels': 183, 'boxes': 303, 'vape_boxes': 203, 'lighter_boxes': 100}
{'dataset': 'v4_ablate_A0_real_only', 'split': 'test', 'images': 184, 'labels': 184, 'boxes': 420, 'vape_boxes': 255, 'lighter_boxes': 165}

v4_ablate_S88_scraped
{'dataset': 'v4_ablate_S88_scraped', 'split': 'train', 'images': 925, 'labels': 925, 'boxes': 1440, 'vape_boxes': 993, 'lighter_boxes': 447}
{'dataset': 'v4_ablate_S88_scraped', 'split': 'val', 'images': 183, 'labels': 183, 'boxes': 303, 'vape_boxes': 203, 'lighter_boxes': 100}
{'dataset': 'v4_ablate_S88_scraped', 'split': 'test', 'images': 184, 'labels': 184, 'boxes': 420, 'vape_boxes': 255, 'lighter_boxes': 165}

v4_ablate_S177_scraped
{'dataset': 'v4_ablate_S177_scraped', 'split': 'train', 'images': 1014, 'labels': 1014, 'b

OSError: [Errno 122] Disk quota exceeded

In [50]:
from pathlib import Path
import shutil

ROOT = Path("/workspace/uavape_v4_model_benchmark")
OUT_ROOT = ROOT / "datasets" / "v4_data_ablation_20260610"
ADDONS = ROOT / "addons"

if OUT_ROOT.exists():
    shutil.rmtree(OUT_ROOT)
    print("Removed partial ablation datasets:", OUT_ROOT)

for z in [ADDONS / "scraped.zip", ADDONS / "synthetic.zip"]:
    if z.exists():
        z.unlink()
        print("Removed zip:", z)

Removed partial ablation datasets: /workspace/uavape_v4_model_benchmark/datasets/v4_data_ablation_20260610
Removed zip: /workspace/uavape_v4_model_benchmark/addons/scraped.zip
Removed zip: /workspace/uavape_v4_model_benchmark/addons/synthetic.zip


In [51]:
from pathlib import Path
import random, shutil, csv, yaml, os
from collections import Counter

def link_file(src, dst):
    src = Path(src).resolve()
    dst = Path(dst)
    dst.parent.mkdir(parents=True, exist_ok=True)
    if dst.exists() or dst.is_symlink():
        dst.unlink()
    try:
        os.link(src, dst)  # no extra disk if same filesystem
    except OSError:
        os.symlink(src, dst)  # fallback, also no image copy

def sample_nested_pairs(pool_dir, doses, seed=SEED):
    pairs = list_yolo_pairs(pool_dir)
    rng = random.Random(seed)
    shuffled = pairs[:]
    rng.shuffle(shuffled)
    selected = shuffled[:max(doses)]
    return {n: sorted(selected[:n], key=lambda p: p[0].name) for n in doses}

def copy_split(src_dataset, dst_dataset, split):
    for kind in ["images", "labels"]:
        src = src_dataset / kind / split
        dst = dst_dataset / kind / split
        dst.mkdir(parents=True, exist_ok=True)
        for item in sorted(src.iterdir()):
            if item.is_file():
                link_file(item, dst / item.name)

def copy_addons(dst_dataset, pairs, prefix):
    rows = []
    for img, lab in pairs:
        img_name = f"{prefix}_{img.name}"
        lab_name = f"{prefix}_{lab.name}"
        link_file(img, dst_dataset / "images" / "train" / img_name)
        link_file(lab, dst_dataset / "labels" / "train" / lab_name)
        rows.append({"source": prefix, "image": str(img), "label": str(lab), "copied_image": img_name})
    return rows

def count_split(dataset, split):
    image_dir = dataset / "images" / split
    label_dir = dataset / "labels" / split
    image_count = len([p for p in image_dir.iterdir() if p.suffix.lower() in IMG_EXTS])
    label_count = len([p for p in label_dir.iterdir() if p.suffix.lower() == ".txt"])
    class_counts = Counter()
    box_count = 0
    for lab in label_dir.glob("*.txt"):
        for line in lab.read_text().splitlines():
            if not line.strip():
                continue
            cls = int(float(line.split()[0]))
            class_counts[cls] += 1
            box_count += 1
    return image_count, label_count, box_count, class_counts

def write_dataset_yaml(dataset):
    data = {
        "path": str(dataset),
        "train": "images/train",
        "val": "images/val",
        "test": "images/test",
        "names": {0: "vape", 1: "lighter"},
    }
    with open(dataset / "dataset.yaml", "w") as f:
        yaml.safe_dump(data, f, sort_keys=False)

def build_dataset(name, scraped=None, synthetic=None):
    scraped = scraped or []
    synthetic = synthetic or []
    dst = OUT_ROOT / name
    if dst.exists():
        shutil.rmtree(dst)

    for split in ["train", "val", "test"]:
        copy_split(BASE, dst, split)

    manifest_rows = []
    manifest_rows += copy_addons(dst, scraped, "scraped")
    manifest_rows += copy_addons(dst, synthetic, "synthetic")
    write_dataset_yaml(dst)

    with open(dst / "addon_manifest.csv", "w", newline="") as f:
        writer = csv.DictWriter(f, fieldnames=["source", "image", "label", "copied_image"])
        writer.writeheader()
        writer.writerows(manifest_rows)

    summary_rows = []
    for split in ["train", "val", "test"]:
        images, labels, boxes, class_counts = count_split(dst, split)
        row = {
            "dataset": name,
            "split": split,
            "images": images,
            "labels": labels,
            "boxes": boxes,
            "vape_boxes": class_counts.get(0, 0),
            "lighter_boxes": class_counts.get(1, 0),
        }
        summary_rows.append(row)

    with open(dst / "dataset_summary.csv", "w", newline="") as f:
        writer = csv.DictWriter(f, fieldnames=summary_rows[0].keys())
        writer.writeheader()
        writer.writerows(summary_rows)

    print(f"\n{name}")
    for row in summary_rows:
        print(row)
    return dst

OUT_ROOT.mkdir(parents=True, exist_ok=True)

scraped_samples = sample_nested_pairs(SCRAPED, DOSES, seed=SEED)
synthetic_samples = sample_nested_pairs(SYNTHETIC, DOSES, seed=SEED)

ABLATION_DATASETS = [build_dataset("v4_ablate_A0_real_only")]

for dose in DOSES:
    ABLATION_DATASETS.append(build_dataset(f"v4_ablate_S{dose}_scraped", scraped=scraped_samples[dose]))

for dose in DOSES:
    ABLATION_DATASETS.append(build_dataset(f"v4_ablate_Y{dose}_synthetic", synthetic=synthetic_samples[dose]))

ABLATION_DATASETS.append(
    build_dataset("v4_ablate_S355_Y355_combined", scraped=scraped_samples[355], synthetic=synthetic_samples[355])
)

ABLATIONS = [p.name for p in ABLATION_DATASETS]
print("\nAblations ready:")
for name in ABLATIONS:
    print(" ", name, OUT_ROOT / name / "dataset.yaml")


v4_ablate_A0_real_only
{'dataset': 'v4_ablate_A0_real_only', 'split': 'train', 'images': 837, 'labels': 837, 'boxes': 1317, 'vape_boxes': 870, 'lighter_boxes': 447}
{'dataset': 'v4_ablate_A0_real_only', 'split': 'val', 'images': 183, 'labels': 183, 'boxes': 303, 'vape_boxes': 203, 'lighter_boxes': 100}
{'dataset': 'v4_ablate_A0_real_only', 'split': 'test', 'images': 184, 'labels': 184, 'boxes': 420, 'vape_boxes': 255, 'lighter_boxes': 165}

v4_ablate_S88_scraped
{'dataset': 'v4_ablate_S88_scraped', 'split': 'train', 'images': 925, 'labels': 925, 'boxes': 1440, 'vape_boxes': 993, 'lighter_boxes': 447}
{'dataset': 'v4_ablate_S88_scraped', 'split': 'val', 'images': 183, 'labels': 183, 'boxes': 303, 'vape_boxes': 203, 'lighter_boxes': 100}
{'dataset': 'v4_ablate_S88_scraped', 'split': 'test', 'images': 184, 'labels': 184, 'boxes': 420, 'vape_boxes': 255, 'lighter_boxes': 165}

v4_ablate_S177_scraped
{'dataset': 'v4_ablate_S177_scraped', 'split': 'train', 'images': 1014, 'labels': 1014, 'b

OSError: [Errno 122] Disk quota exceeded

In [52]:
from pathlib import Path
import shutil

ROOT = Path("/workspace/uavape_v4_model_benchmark")
OUT_ROOT = ROOT / "datasets" / "v4_data_ablation_20260610"

if OUT_ROOT.exists():
    shutil.rmtree(OUT_ROOT)
    print("Removed:", OUT_ROOT)

# Optional disk check
!df -h /workspace
!du -sh /workspace/uavape_v4_model_benchmark/* 2>/dev/null | sort -h

Removed: /workspace/uavape_v4_model_benchmark/datasets/v4_data_ablation_20260610
Filesystem                   Size  Used Avail Use% Mounted on
mfs#us-ga-2.runpod.net:9421  825T  548T  277T  67% /workspace
2.5K	/workspace/uavape_v4_model_benchmark/v4_per_class_val_summary.csv
16M	/workspace/uavape_v4_model_benchmark/tiling_eval
906M	/workspace/uavape_v4_model_benchmark/runs
1.5G	/workspace/uavape_v4_model_benchmark/addons
7.7G	/workspace/uavape_v4_model_benchmark/datasets


In [53]:
from pathlib import Path
import random, shutil, csv, yaml, os
from collections import Counter
from ultralytics import YOLO

DATA_ROOT = ROOT / "datasets" / "v4_data_ablation_20260610"
CURRENT = DATA_ROOT / "_current_dataset"
RUNS_ROOT = ROOT / "runs_data_ablation"

MODEL = "yolo26s.pt"
IMGSZ = 1280
EPOCHS = 100
SEED = 42

def link_file(src, dst):
    src = Path(src).resolve()
    dst = Path(dst)
    dst.parent.mkdir(parents=True, exist_ok=True)
    if dst.exists() or dst.is_symlink():
        dst.unlink()
    try:
        os.symlink(src, dst)  # lowest quota pressure
    except OSError:
        os.link(src, dst)

def sample_nested_pairs(pool_dir, doses, seed=SEED):
    pairs = list_yolo_pairs(pool_dir)
    rng = random.Random(seed)
    shuffled = pairs[:]
    rng.shuffle(shuffled)
    selected = shuffled[:max(doses)]
    return {n: sorted(selected[:n], key=lambda p: p[0].name) for n in doses}

def copy_split(src_dataset, dst_dataset, split):
    for kind in ["images", "labels"]:
        src = src_dataset / kind / split
        dst = dst_dataset / kind / split
        dst.mkdir(parents=True, exist_ok=True)
        for item in sorted(src.iterdir()):
            if item.is_file():
                link_file(item, dst / item.name)

def copy_addons(dst_dataset, pairs, prefix):
    rows = []
    for img, lab in pairs:
        img_name = f"{prefix}_{img.name}"
        lab_name = f"{prefix}_{lab.name}"
        link_file(img, dst_dataset / "images" / "train" / img_name)
        link_file(lab, dst_dataset / "labels" / "train" / lab_name)
        rows.append({"source": prefix, "image": str(img), "label": str(lab), "copied_image": img_name})
    return rows

def write_dataset_yaml(dataset):
    data = {
        "path": str(dataset),
        "train": "images/train",
        "val": "images/val",
        "test": "images/test",
        "names": {0: "vape", 1: "lighter"},
    }
    with open(dataset / "dataset.yaml", "w") as f:
        yaml.safe_dump(data, f, sort_keys=False)

def count_split(dataset, split):
    image_dir = dataset / "images" / split
    label_dir = dataset / "labels" / split
    image_count = len([p for p in image_dir.iterdir() if p.suffix.lower() in IMG_EXTS])
    label_count = len([p for p in label_dir.iterdir() if p.suffix.lower() == ".txt"])
    class_counts = Counter()
    box_count = 0
    for lab in label_dir.glob("*.txt"):
        for line in lab.read_text().splitlines():
            if not line.strip():
                continue
            cls = int(float(line.split()[0]))
            class_counts[cls] += 1
            box_count += 1
    return image_count, label_count, box_count, class_counts

def build_current_dataset(run_name, scraped=None, synthetic=None):
    scraped = scraped or []
    synthetic = synthetic or []

    if CURRENT.exists():
        shutil.rmtree(CURRENT)
    CURRENT.mkdir(parents=True, exist_ok=True)

    for split in ["train", "val", "test"]:
        copy_split(BASE, CURRENT, split)

    manifest_rows = []
    manifest_rows += copy_addons(CURRENT, scraped, "scraped")
    manifest_rows += copy_addons(CURRENT, synthetic, "synthetic")
    write_dataset_yaml(CURRENT)

    with open(CURRENT / "addon_manifest.csv", "w", newline="") as f:
        writer = csv.DictWriter(f, fieldnames=["source", "image", "label", "copied_image"])
        writer.writeheader()
        writer.writerows(manifest_rows)

    print("\n", run_name)
    for split in ["train", "val", "test"]:
        images, labels, boxes, class_counts = count_split(CURRENT, split)
        print({
            "split": split,
            "images": images,
            "labels": labels,
            "boxes": boxes,
            "vape_boxes": class_counts.get(0, 0),
            "lighter_boxes": class_counts.get(1, 0),
        })

    return CURRENT / "dataset.yaml"

DATA_ROOT.mkdir(parents=True, exist_ok=True)
RUNS_ROOT.mkdir(parents=True, exist_ok=True)

scraped_samples = sample_nested_pairs(SCRAPED, DOSES, seed=SEED)
synthetic_samples = sample_nested_pairs(SYNTHETIC, DOSES, seed=SEED)

ABLATION_SPECS = [
    ("v4_ablate_A0_real_only", [], []),
    ("v4_ablate_S88_scraped", scraped_samples[88], []),
    ("v4_ablate_S177_scraped", scraped_samples[177], []),
    ("v4_ablate_S355_scraped", scraped_samples[355], []),
    ("v4_ablate_Y88_synthetic", [], synthetic_samples[88]),
    ("v4_ablate_Y177_synthetic", [], synthetic_samples[177]),
    ("v4_ablate_Y355_synthetic", [], synthetic_samples[355]),
    ("v4_ablate_S355_Y355_combined", scraped_samples[355], synthetic_samples[355]),
]

for run_name, scraped, synthetic in ABLATION_SPECS:
    data_yaml = build_current_dataset(run_name, scraped=scraped, synthetic=synthetic)

    model = YOLO(MODEL)
    model.train(
        data=str(data_yaml),
        imgsz=IMGSZ,
        epochs=EPOCHS,
        batch=16,
        seed=SEED,
        deterministic=True,
        patience=100,
        project=str(RUNS_ROOT),
        name=run_name,
        exist_ok=True,
        cos_lr=True,
        device=0,
        workers=8,
        val=True,
    )

    # Preserve the exact add-on manifest beside the run before deleting temp dataset.
    run_dir = RUNS_ROOT / run_name
    shutil.copy2(CURRENT / "dataset.yaml", run_dir / "dataset.yaml")
    shutil.copy2(CURRENT / "addon_manifest.csv", run_dir / "addon_manifest.csv")

    shutil.rmtree(CURRENT)
    print("Deleted temporary dataset:", CURRENT)


 v4_ablate_A0_real_only
{'split': 'train', 'images': 837, 'labels': 837, 'boxes': 1317, 'vape_boxes': 870, 'lighter_boxes': 447}
{'split': 'val', 'images': 183, 'labels': 183, 'boxes': 303, 'vape_boxes': 203, 'lighter_boxes': 100}
{'split': 'test', 'images': 184, 'labels': 184, 'boxes': 420, 'vape_boxes': 255, 'lighter_boxes': 165}
New https://pypi.org/project/ultralytics/8.4.64 available 😃 Update with 'pip install -U ultralytics'
Ultralytics 8.4.62 🚀 Python-3.12.3 torch-2.8.0+cu128 CUDA:0 (NVIDIA H100 80GB HBM3, 81079MiB)
engine/trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=16, bgr=0.0, box=7.5, cache=False, cfg=None, classes=None, close_mosaic=10, cls=0.5, cls_pw=0.0, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=True, cutmix=0.0, data=/workspace/uavape_v4_model_benchmark/datasets/v4_data_ablation_20260610/_current_dataset/dataset.yaml, degrees=0.0, deterministic=True, device=0, dfl=1.5, dnn=False, dropout=0

KeyboardInterrupt: 

In [54]:
from pathlib import Path
import shutil

ROOT = Path("/workspace/uavape_v4_model_benchmark")
CURRENT = ROOT / "datasets" / "v4_data_ablation_20260610" / "_current_dataset"

if CURRENT.exists():
    shutil.rmtree(CURRENT)
    print("Deleted temp dataset:", CURRENT)
else:
    print("No temp dataset found.")

Deleted temp dataset: /workspace/uavape_v4_model_benchmark/datasets/v4_data_ablation_20260610/_current_dataset


In [55]:
A0_RUN = ROOT / "runs_data_ablation" / "v4_ablate_A0_real_only"

if A0_RUN.exists() and not (A0_RUN / "weights" / "best.pt").exists():
    shutil.rmtree(A0_RUN)
    print("Deleted incomplete A0 run:", A0_RUN)
else:
    print("No incomplete A0 run to delete, or it already completed.")

No incomplete A0 run to delete, or it already completed.


In [56]:
from pathlib import Path
import random, shutil, csv, yaml, os
from collections import Counter
from ultralytics import YOLO

DATA_ROOT = ROOT / "datasets" / "v4_data_ablation_20260610"
CURRENT = DATA_ROOT / "_current_dataset"
RUNS_ROOT = ROOT / "runs_data_ablation"

MODEL = "yolo26s.pt"
IMGSZ = 1280
EPOCHS = 100
SEED = 42

def link_file(src, dst):
    src = Path(src).resolve()
    dst = Path(dst)
    dst.parent.mkdir(parents=True, exist_ok=True)
    if dst.exists() or dst.is_symlink():
        dst.unlink()
    os.symlink(src, dst)

def sample_nested_pairs(pool_dir, doses, seed=SEED):
    pairs = list_yolo_pairs(pool_dir)
    rng = random.Random(seed)
    shuffled = pairs[:]
    rng.shuffle(shuffled)
    selected = shuffled[:max(doses)]
    return {n: sorted(selected[:n], key=lambda p: p[0].name) for n in doses}

def copy_split(src_dataset, dst_dataset, split):
    for kind in ["images", "labels"]:
        src = src_dataset / kind / split
        dst = dst_dataset / kind / split
        dst.mkdir(parents=True, exist_ok=True)
        for item in sorted(src.iterdir()):
            if item.is_file():
                link_file(item, dst / item.name)

def copy_addons(dst_dataset, pairs, prefix):
    rows = []
    for img, lab in pairs:
        img_name = f"{prefix}_{img.name}"
        lab_name = f"{prefix}_{lab.name}"
        link_file(img, dst_dataset / "images" / "train" / img_name)
        link_file(lab, dst_dataset / "labels" / "train" / lab_name)
        rows.append({"source": prefix, "image": str(img), "label": str(lab), "copied_image": img_name})
    return rows

def write_dataset_yaml(dataset):
    data = {
        "path": str(dataset),
        "train": "images/train",
        "val": "images/val",
        "test": "images/test",
        "names": {0: "vape", 1: "lighter"},
    }
    with open(dataset / "dataset.yaml", "w") as f:
        yaml.safe_dump(data, f, sort_keys=False)

def count_split(dataset, split):
    image_dir = dataset / "images" / split
    label_dir = dataset / "labels" / split
    image_count = len([p for p in image_dir.iterdir() if p.suffix.lower() in IMG_EXTS])
    label_count = len([p for p in label_dir.iterdir() if p.suffix.lower() == ".txt"])
    class_counts = Counter()
    box_count = 0

    for lab in label_dir.glob("*.txt"):
        for line in lab.read_text().splitlines():
            if not line.strip():
                continue
            cls = int(float(line.split()[0]))
            class_counts[cls] += 1
            box_count += 1

    return image_count, label_count, box_count, class_counts

def build_current_dataset(run_name, scraped=None, synthetic=None):
    scraped = scraped or []
    synthetic = synthetic or []

    if CURRENT.exists():
        shutil.rmtree(CURRENT)
    CURRENT.mkdir(parents=True, exist_ok=True)

    for split in ["train", "val", "test"]:
        copy_split(BASE, CURRENT, split)

    manifest_rows = []
    manifest_rows += copy_addons(CURRENT, scraped, "scraped")
    manifest_rows += copy_addons(CURRENT, synthetic, "synthetic")

    write_dataset_yaml(CURRENT)

    with open(CURRENT / "addon_manifest.csv", "w", newline="") as f:
        writer = csv.DictWriter(f, fieldnames=["source", "image", "label", "copied_image"])
        writer.writeheader()
        writer.writerows(manifest_rows)

    print("\n" + "=" * 80)
    print(run_name)
    print("=" * 80)

    for split in ["train", "val", "test"]:
        images, labels, boxes, class_counts = count_split(CURRENT, split)
        print({
            "split": split,
            "images": images,
            "labels": labels,
            "boxes": boxes,
            "vape_boxes": class_counts.get(0, 0),
            "lighter_boxes": class_counts.get(1, 0),
        })

    return CURRENT / "dataset.yaml"

DATA_ROOT.mkdir(parents=True, exist_ok=True)
RUNS_ROOT.mkdir(parents=True, exist_ok=True)

scraped_samples = sample_nested_pairs(SCRAPED, DOSES, seed=SEED)
synthetic_samples = sample_nested_pairs(SYNTHETIC, DOSES, seed=SEED)

ABLATION_SPECS = [
    ("v4_ablate_S88_scraped", scraped_samples[88], []),
    ("v4_ablate_S177_scraped", scraped_samples[177], []),
    ("v4_ablate_S355_scraped", scraped_samples[355], []),
    ("v4_ablate_Y88_synthetic", [], synthetic_samples[88]),
    ("v4_ablate_Y177_synthetic", [], synthetic_samples[177]),
    ("v4_ablate_Y355_synthetic", [], synthetic_samples[355]),
    ("v4_ablate_S355_Y355_combined", scraped_samples[355], synthetic_samples[355]),
]

for run_name, scraped, synthetic in ABLATION_SPECS:
    run_dir = RUNS_ROOT / run_name
    if (run_dir / "weights" / "best.pt").exists():
        print("Skipping completed run:", run_name)
        continue

    data_yaml = build_current_dataset(run_name, scraped=scraped, synthetic=synthetic)

    model = YOLO(MODEL)
    model.train(
        data=str(data_yaml),
        imgsz=IMGSZ,
        epochs=EPOCHS,
        batch=16,
        seed=SEED,
        deterministic=True,
        patience=100,
        project=str(RUNS_ROOT),
        name=run_name,
        exist_ok=True,
        cos_lr=True,
        device=0,
        workers=8,
        val=True,
    )

    run_dir.mkdir(parents=True, exist_ok=True)
    shutil.copy2(CURRENT / "dataset.yaml", run_dir / "dataset.yaml")
    shutil.copy2(CURRENT / "addon_manifest.csv", run_dir / "addon_manifest.csv")

    shutil.rmtree(CURRENT)
    print("Deleted temporary dataset:", CURRENT)


v4_ablate_S88_scraped
{'split': 'train', 'images': 925, 'labels': 925, 'boxes': 1440, 'vape_boxes': 993, 'lighter_boxes': 447}
{'split': 'val', 'images': 183, 'labels': 183, 'boxes': 303, 'vape_boxes': 203, 'lighter_boxes': 100}
{'split': 'test', 'images': 184, 'labels': 184, 'boxes': 420, 'vape_boxes': 255, 'lighter_boxes': 165}
New https://pypi.org/project/ultralytics/8.4.64 available 😃 Update with 'pip install -U ultralytics'
Ultralytics 8.4.62 🚀 Python-3.12.3 torch-2.8.0+cu128 CUDA:0 (NVIDIA H100 80GB HBM3, 81079MiB)
engine/trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=16, bgr=0.0, box=7.5, cache=False, cfg=None, classes=None, close_mosaic=10, cls=0.5, cls_pw=0.0, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=True, cutmix=0.0, data=/workspace/uavape_v4_model_benchmark/datasets/v4_data_ablation_20260610/_current_dataset/dataset.yaml, degrees=0.0, deterministic=True, device=0, dfl=1.5, dnn=False, dropout=0.0